In [1]:
import sys
import os

# Add the src/ directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "src/")))

In [2]:
from feature_extraction import processEpoch, processSub
from schema_definition import get_feature_schema, get_subject_schema
from preprocess_sets import subPath, participantsInfoPath, processSubPSDs, processSub

In [3]:
# test_imports.py
from preprocess_sets import subPath, participantsInfoPath, processSubPSDs, processSub

# Test 1: Check if functions are imported correctly
print("Function check:")
print("subPath exists:", subPath is not None)
print("participantsInfoPath exists:", participantsInfoPath is not None)
print("processSubPSDs exists:", processSubPSDs is not None)
print("processSub exists:", processSub is not None)

# Test 2: Check if path functions return expected values
print("\nPath check:")
try:
    participants_path = participantsInfoPath()
    print("Participants info path:", participants_path)
    print("Path exists:", os.path.exists(participants_path))
except Exception as e:
    print("Error getting participants path:", str(e))

# Test 3: Check if subPath works for a sample subject
print("\nsubPath check:")
try:
    subject_id = "001"  # Change to a subject ID you know exists
    path = subPath(subject_id, derivatives=True)
    print(f"Path for subject {subject_id}:", path)
    print("Path exists:", os.path.exists(path))
except Exception as e:
    print(f"Error getting path for subject {subject_id}:", str(e))

# Only run this if you're confident the above tests passed
# as this will attempt to actually load data
print("\nMini processSub check:")
try:
    import time
    start = time.time()
    subject_id = "001"  # Use a known subject ID
    print(f"Attempting to get first epoch for {subject_id}...")
    epochs = processSub(subject_id)
    print(f"Got {len(epochs)} epochs in {time.time() - start:.2f} seconds")
    print("First epoch shape:", epochs[0].get_data().shape)
except Exception as e:
    print(f"Error processing subject {subject_id}:", str(e))
print("check end")

Function check:
subPath exists: True
participantsInfoPath exists: True
processSubPSDs exists: True
processSub exists: True

Path check:
Participants info path: /Users/user/eeg-ds004504/ds004504/participants.tsv
Path exists: True

subPath check:
subPath 001
Path for subject 001: /Users/user/eeg-ds004504/ds004504/derivatives/sub-001/eeg/sub-001_task-eyesclosed_eeg.set
Path exists: True

Mini processSub check:
Attempting to get first epoch for 001...
processSub 001
subPath 001
Got 398 epochs in 1.82 seconds
First epoch shape: (1, 19, 1501)


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
from schema_definition import get_subject_schema, get_feature_schema
from feature_extraction import processEpoch, processSub
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import DataFrame

In [5]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import DataFrame


def load_subjects_df(spark: SparkSession, participants_path: str) -> DataFrame:
    """
    Reads participants.tsv and returns a Spark DataFrame
    with columns SubjectID and Group for groups A, C, and F.

    Parameters:
        spark (SparkSession): Active Spark session
        participants_path (str): Path to the participants.tsv file

    Returns:
        Spark DataFrame with SubjectID and Group columns
    """
    participantsInfo = pd.read_table(participants_path)

    records = []
    for group_code in ["A", "C", "F"]:
        group_subjects = participantsInfo[participantsInfo["Group"] == group_code]["participant_id"].tolist()
        for sub in group_subjects:
            records.append((sub, group_code))
    return spark.createDataFrame(records, schema=get_subject_schema())


In [6]:
spark = SparkSession.builder.appName("MyApp").getOrCreate()

subject_df = load_subjects_df(spark, "ds004504/participants.tsv")

# subjects_df.show()
subject_df.show(n=subject_df.count(), truncate=False)# Optionally, you can save this DataFrame

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/27 18:27:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/03/27 18:27:44 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/03/27 18:27:44 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/03/27 18:27:44 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
                                                                                

+---------+-----+
|SubjectID|Group|
+---------+-----+
|sub-001  |A    |
|sub-002  |A    |
|sub-003  |A    |
|sub-004  |A    |
|sub-005  |A    |
|sub-006  |A    |
|sub-007  |A    |
|sub-008  |A    |
|sub-009  |A    |
|sub-010  |A    |
|sub-011  |A    |
|sub-012  |A    |
|sub-013  |A    |
|sub-014  |A    |
|sub-015  |A    |
|sub-016  |A    |
|sub-017  |A    |
|sub-018  |A    |
|sub-019  |A    |
|sub-020  |A    |
|sub-021  |A    |
|sub-022  |A    |
|sub-023  |A    |
|sub-024  |A    |
|sub-025  |A    |
|sub-026  |A    |
|sub-027  |A    |
|sub-028  |A    |
|sub-029  |A    |
|sub-030  |A    |
|sub-031  |A    |
|sub-032  |A    |
|sub-033  |A    |
|sub-034  |A    |
|sub-035  |A    |
|sub-036  |A    |
|sub-037  |C    |
|sub-038  |C    |
|sub-039  |C    |
|sub-040  |C    |
|sub-041  |C    |
|sub-042  |C    |
|sub-043  |C    |
|sub-044  |C    |
|sub-045  |C    |
|sub-046  |C    |
|sub-047  |C    |
|sub-048  |C    |
|sub-049  |C    |
|sub-050  |C    |
|sub-051  |C    |
|sub-052  |C    |
|sub-053  

In [7]:
subject_df.printSchema()

root
 |-- SubjectID: string (nullable = false)
 |-- Group: string (nullable = false)



In [8]:
subject_df.filter(subject_df.SubjectID == "sub-001").show()


+---------+-----+
|SubjectID|Group|
+---------+-----+
|  sub-001|    A|
+---------+-----+



# Getting Spark Function to work

In [9]:
# Get the SparkContext from your existing SparkSession
sc = spark.sparkContext

# Add your Python modules to all workers
import os

# For Jupyter notebook, use relative path to src directory
script_dir = os.path.abspath(os.path.join(os.getcwd(), "../src"))
print(f"Using source directory: {script_dir}")

# Check if the directory exists
if not os.path.exists(script_dir):
    print(f"Warning: Directory {script_dir} does not exist!")
    # Fallback to alternative paths
    possible_paths = [
        os.path.abspath(os.path.join(os.getcwd(), "src")),
        os.path.abspath(os.path.join(os.getcwd(), "../src")),
        os.path.abspath(os.path.join(os.getcwd(), "../../src"))
    ]
    
    for path in possible_paths:
        if os.path.exists(path):
            print(f"Found alternative path: {path}")
            script_dir = path
            break
    else:
        print("Could not find src directory. Please specify the full path.")

# List files in the directory to verify
print("Files in the directory:")
try:
    for file in os.listdir(script_dir):
        if file.endswith('.py'):
            print(f"  - {file}")
except Exception as e:
    print(f"Error listing directory: {e}")

# Add all necessary modules
try:
    sc.addPyFile(os.path.join(script_dir, "feature_extraction.py"))
    print("Added feature_extraction.py")
    sc.addPyFile(os.path.join(script_dir, "preprocess_sets.py"))
    print("Added preprocess_sets.py")
    sc.addPyFile(os.path.join(script_dir, "schema_definition.py"))
    print("Added schema_definition.py")
except Exception as e:
    print(f"Error adding files to SparkContext: {e}")

Using source directory: /Users/user/src
Found alternative path: /Users/user/eeg-ds004504/src
Files in the directory:
  - populate_schemas.py
  - preprocess_sets.py
  - __init__.py
  - test.py
  - feature_extraction.py
  - schema_definition.py
Added feature_extraction.py
Added preprocess_sets.py
Added schema_definition.py


In [ ]:
@pandas_udf(get_feature_schema(), PandasUDFType.GROUPED_MAP)
def extract_features_udtf(pdf):
    from feature_extraction import processEpoch, processSub
    from schema_definition import get_feature_schema, get_subject_schema
    print("function ran")
    rows = []
    for _, row in pdf.iterrows():
        subject_id = row["SubjectID"]
        try:
            epochs = processSub(subject_id, derivatives=False)
            for i, epoch in enumerate(epochs):
                epoch_id = f"ep-{i}"
                features = processEpoch(epoch)
                for (electrode, band), stats in features:
                    # Assuming `stats` is a tuple with (mean, variance, skewness, kurtosis)
                    rows.append((subject_id, epoch_id, band, electrode, *stats))
        except Exception as e:
            print(f"Error processing {subject_id}: {e}")
    return pd.DataFrame(rows, columns=[f.name for f in get_feature_schema()])



In [ ]:
# debugging funcion

In [40]:
@pandas_udf(get_feature_schema(), PandasUDFType.GROUPED_MAP)
def extract_features_udtf(pdf):
    from feature_extraction import processEpoch, processSub
    from schema_definition import get_feature_schema, get_subject_schema
    import mne
    print("function ran")
    rows = []
    for _, row in pdf.iterrows():
        subject_id = row["SubjectID"]
        try:
            print(f"Processing subject {subject_id}")
            epochs = processSub(subject_id, derivatives=False)
            print(f"Got {len(epochs)} epochs for {subject_id}")
            print(type(epochs)) 
            # for i, epoch in enumerate(epochs):
            for i in range(len(epochs)):
                epoch = epochs[i]
                if i < 2:  # Just print info for the first 2 epochs to avoid spam
                    # print(f"Epoch {i} shape: {epoch.to_data_frame().shape()}")
                    print(f"Epoch {i}")
                    print(type(epoch))
                    print(type(epochs[i]))
                epoch_id = f"ep-{i}"
                features = processEpoch(epoch)
                
                if i < 2:  # Debug output
                    print(f"Epoch {i} features count: {len(features) if features else 0}")
                    if features and len(features) > 0:
                        print(f"First feature sample: {next(iter(features))}")
                
                for item in features:
                    # Check the structure of each item
                    electrode_band_key, stats_value = item
                    electrode, band = electrode_band_key
                    print(f"Adding: {subject_id}, {epoch_id}, {band}, {electrode}, stats: {stats_value}")
                    
                    # Add to results - adjust this based on actual structure 
                    # TODO : **Error here ! have to line it up! :)
                    try:
                        rows.append((subject_id, epoch_id, band, electrode, *stats_value))
                    except Exception as e:
                        print(f"Error appending row: {e}, stats_value: {stats_value}")
            
            print(f"Total rows collected: {len(rows)}")
            
        except Exception as e:
            print(f"Error processing {subject_id}: {e}")
            import traceback
            traceback.print_exc()
            
    # Print final row count before returning
    print(f"Returning DataFrame with {len(rows)} rows")
    
    # Check if we have column names from schema
    schema_fields = get_feature_schema()
    column_names = [f.name for f in schema_fields]
    print(f"Column names from schema: {column_names}")
    
    import pandas as pd
    result_df = pd.DataFrame(rows, columns=column_names)
    print(f"Result DataFrame shape: {result_df.shape}")
    return result_df

In [41]:
print(processSub('sub-001', derivatives=False))

processSub sub-001
subPath sub-001
<Epochs | 398 events (all good), 0 – 3 s (baseline off), ~86.6 MiB, data loaded,
 '1': 398>


In [42]:
# testing a single subject

In [43]:
result = (
    subject_df
    .filter((subject_df.SubjectID == "sub-001") & (subject_df.Group == "A"))
    .groupBy("SubjectID")
    .apply(extract_features_udtf)
)

result.show()

/Users/user/jupyter-venv/lib/python3.12/site-packages/pyspark/sql/pandas/group_ops.py:104: UserWarning: It is preferred to use 'applyInPandas' over this API. This API will be deprecated in the future releases. See SPARK-28264 for more details.
  warnings.warn(
function ran
Processing subject sub-001
processSub sub-001
subPath sub-001
Got 398 epochs for sub-001                                          (0 + 1) / 1]
<class 'mne.epochs.Epochs'>
Epoch 0
<class 'mne.epochs.Epochs'>
<class 'mne.epochs.Epochs'>
Epoch 0 features count: 76
First feature sample: (('Fp1', 'Delta'), np.float64(0.08257892642542035))
Adding: sub-001, ep-0, Delta, Fp1, stats: 0.08257892642542035
Error appending row: Value after * must be an iterable, not numpy.float64, stats_value: 0.08257892642542035
Adding: sub-001, ep-0, Delta, Fp2, stats: 0.08389220643958976
Error appending row: Value after * must be an iterable, not numpy.float64, stats_value: 0.08389220643958976
Adding: sub-001, ep-0, Delta, F3, stats: 0.0821227

+---------+-------+--------+---------+-----+--------+--------+--------+---+---+----+
|SubjectID|EpochID|WaveBand|Electrode|Power|Skewness|Kurtosis|Variance|Min|Max|Mean|
+---------+-------+--------+---------+-----+--------+--------+--------+---+---+----+
+---------+-------+--------+---------+-----+--------+--------+--------+---+---+----+



Adding: sub-001, ep-396, Delta, Fp1, stats: 0.07162250528412895
Error appending row: Value after * must be an iterable, not numpy.float64, stats_value: 0.07162250528412895
Adding: sub-001, ep-396, Delta, Fp2, stats: 0.07077830772056051
Error appending row: Value after * must be an iterable, not numpy.float64, stats_value: 0.07077830772056051
Adding: sub-001, ep-396, Delta, F3, stats: 0.07311055027878154
Error appending row: Value after * must be an iterable, not numpy.float64, stats_value: 0.07311055027878154
Adding: sub-001, ep-396, Delta, F4, stats: 0.06878926094336331
Error appending row: Value after * must be an iterable, not numpy.float64, stats_value: 0.06878926094336331
Adding: sub-001, ep-396, Delta, C3, stats: 0.07317768586781274
Error appending row: Value after * must be an iterable, not numpy.float64, stats_value: 0.07317768586781274
Adding: sub-001, ep-396, Delta, C4, stats: 0.07069663602140666
Error appending row: Value after * must be an iterable, not numpy.float64, stats

In [ ]:
 # testing everything 

In [ ]:
result = subject_df.groupBy("Group").apply(extract_features_udtf)
result.show()

In [ ]:
result = subject_df.filter((subject_df.SubjectID == "sub-001") & (subject_df.Group == "A")).show()
result